In [ ]:
import numpy as np
import os
import pandas as pd
from datetime import date, datetime
import matplotlib.pyplot as plt
import shutil
import json
import math

## Data treatment
### Functions

In [ ]:
# Adding the failures...
# In his approach we decided to use np.nan, not -1
def data_treatment_intervals(destination_data_path, original_data_path, protocols):
    for protocol in protocols:
        data_path = os.path.join(original_data_path, protocol)
        destination_path = os.path.join(destination_data_path, protocol)

        files = os.listdir(data_path)
        sorted_files = sorted(files)

        if not os.path.exists(destination_path):
            os.makedirs(destination_path)

        for file_name in sorted_files:
            print(file_name)
            dataset = pd.read_csv(os.path.join(data_path, file_name))

            if dataset.empty:
                print(f"No data in {file_name}. Skipping...")
                continue
            
            dataset = dataset.rename(columns={' Vazao': 'Throughput'})

            # Ensure all Timestamp values are numeric before conversion
            dataset = dataset[pd.to_numeric(dataset['Timestamp'], errors='coerce').notna()]
            dataset['Timestamp'] = dataset['Timestamp'].astype(float)
            dataset['Timestamp'] = pd.to_datetime(dataset['Timestamp'], unit='s')
            
            # Define function to get interval start time
            def get_interval_start(dt):
                hour = dt.hour
                if hour < 6:
                    return dt.replace(hour=0, minute=0, second=0, microsecond=0)
                elif hour < 12:
                    return dt.replace(hour=6, minute=0, second=0, microsecond=0)
                elif hour < 18:
                    return dt.replace(hour=12, minute=0, second=0, microsecond=0)
                else:
                    return dt.replace(hour=18, minute=0, second=0, microsecond=0)

            # Apply the function to create 'Interval' column
            dataset['Timestamp'] = dataset['Timestamp'].apply(get_interval_start)

            def calculate_mean_without_outliers(data):
                Q1 = data.quantile(0.25)
                Q3 = data.quantile(0.75)
                IQR = Q3 - Q1
                lower_outlier_cut = Q1 - 1.5 * IQR
                upper_outlier_cut = Q3 + 1.5 * IQR
                filtered_data = data[(data >= lower_outlier_cut) & (data <= upper_outlier_cut)]
                return filtered_data.mean()

            # Group by 'Interval' and calculate mean of 'Throughput'
            grouped = dataset.groupby('Timestamp')['Throughput'].apply(calculate_mean_without_outliers).reset_index()

            start_time = dataset['Timestamp'].min().replace(hour=0, minute=0, second=0, microsecond=0)
            end_time = dataset['Timestamp'].max().replace(hour=18, minute=0, second=0, microsecond=0)
            all_intervals = pd.date_range(start=start_time, end=end_time, freq='6h')

            grouped = grouped.set_index('Timestamp').reindex(all_intervals).reset_index()
            grouped.columns = ['Timestamp', 'Throughput']  # Rename columns after reindex
            grouped['Throughput'] = grouped['Throughput'].fillna(np.nan)

            # Format 'Interval' to 'dd-mm-aa hh:mm:ss'
            grouped['Timestamp'] = grouped['Timestamp'].dt.strftime('%d-%m-%y %H:%M:%S')

            output_file = os.path.join(destination_path, "treated " + file_name)
            grouped.to_csv(output_file, index=False)

def plot_original(df, ax, name, column_name=' Vazao', time_column_name='Timestamp', point_size = 2, point_color = 'red'):
    df[time_column_name] = pd.to_datetime(df[time_column_name])
    df = df.sort_values(by=time_column_name)

    ax.scatter(df[time_column_name], df[column_name], s=point_size, c=point_color)  
    ax.set_xlabel('Timestamp')
    ax.set_ylabel('Throughput')
    ax.set_title(name)
    ax.tick_params(axis='x', rotation=45)

## Checking how many data we lost by cleaning outliers and applying interval 
def generate_plots(link, original_data_path, treated_path):
    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'Throughput by Time for {link}', fontsize=16)

    # Original BBR
    df = pd.read_csv(f'{original_data_path}/bbr/bbr {link}.csv')
    plot_original(df, axs[0, 0], f'Original BBR {link}')

    # Original Cubic
    df2 = pd.read_csv(f'{original_data_path}/cubic/cubic {link}.csv')
    plot_original(df2, axs[0, 1], f'Original Cubic {link}')

    # Treated BBR
    df = pd.read_csv(f'{treated_path}/bbr/treated bbr {link}.csv')
    df = df.dropna(subset=['Throughput'])
    plot_original(df, axs[1, 0], f'Treated BBR {link}', 'Throughput')

    # Treated Cubic
    df2 = pd.read_csv(f'{treated_path}/cubic/treated cubic {link}.csv')
    df2 = df2.dropna(subset=['Throughput']) 
    plot_original(df2, axs[1, 1], f'Treated Cubic {link}', 'Throughput')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  
    plt.show()

## Dataset analysis
### Functions
Analysing failure rate and size of datasets collection with MonIpê Network Monitoring Tool

In [ ]:
def get_longest_interval(treated_throughput_path, saving_path='longest interval', quantity=10):
    all_files = []

    for protocol in os.listdir(treated_throughput_path):
        folder_path = os.path.join(treated_throughput_path, protocol)
        if os.path.isdir(folder_path):  # Ensure it's a directory
            for file in os.listdir(folder_path):
                file_path = os.path.join(folder_path, file)
                if file.endswith('.csv'):
                    df = pd.read_csv(file_path)

                    longest_interval = []
                    current_interval = []

                    for _, row in df.iterrows():
                        if not pd.isna(row['Throughput']):  # Check for non-NaN values
                            current_interval.append(row)
                        else:
                            if len(current_interval) > len(longest_interval):
                                longest_interval = current_interval
                            current_interval = []

                    # Final check after loop to capture the last interval
                    if len(current_interval) > len(longest_interval):
                        longest_interval = current_interval

                    # Store file and interval information
                    all_files.append({'file': file, 'interval_length': len(longest_interval)})

    # Sort by interval length and get the top files based on quantity
    sorted_files = sorted(all_files, key=lambda x: x['interval_length'], reverse=True)[:quantity]

    # Save only the top `quantity` longest intervals
    if not os.path.exists(saving_path):
        os.makedirs(saving_path)

    for entry in sorted_files:
        longest_df = pd.DataFrame(entry['data'])
        output_file = os.path.splitext(entry['file'])[0] + '_longest_interval.csv'
        longest_df.to_csv(os.path.join(saving_path, output_file), index=False)

    return sorted_files

# This function gets the greater longest interval (pre defined in quantity parameter)
def get_files_size_failure_rate(percentage, path, protocols, quantity=10):
    archive_data = {}
    file_list = []
    file_list_names = []

    # Loop through all relevant CSV files in the directory
    for protocol in protocols:
        full_path = os.path.join(path, protocol)
        for arquivo in os.listdir(full_path):
            if arquivo.endswith('.csv'):
                caminho_arquivo = os.path.join(full_path, arquivo)
                df = pd.read_csv(caminho_arquivo)

                total_rows = len(df)
                # Check if the dataset has more than 700 lines
                if total_rows <= 700:
                    continue  # Skip the file if it has 700 lines or less

                num_failures = df['Throughput'].isna().sum()
                failure_percentage = (num_failures / total_rows) * 100 if total_rows > 0 else 0
                
                # Include only archives with failure percentage below specified percentage
                if failure_percentage < percentage:
                    # Store the DataFrame, total rows, and failure percentage in the dictionary
                    archive_data[f'{protocol} {arquivo}'] = {'df': df, 'total_rows': total_rows, 'failure_percentage': failure_percentage}
                    file_list_names.append(arquivo)

    # Check if any archives meet the criteria
    if not archive_data:
        print(f"No archives with failure percentage below {percentage}% were found.")
        return []

    # Sort the archives by line count (descending) and then by failure percentage (ascending)
    sorted_archives = sorted(
        archive_data.items(),
        key=lambda x: (-x[1]['total_rows'], x[1]['failure_percentage'])
    )

    top_archives = sorted_archives[:quantity]

    print(f"Top {quantity} archives with failure percentage below {percentage}% and more than 700 lines:")
    for archive, data in top_archives:
        failure_percentage = data['failure_percentage']
        total_rows = data['total_rows']
        file_list.append(f"{archive}: {total_rows} lines, Failure Percentage: {failure_percentage:.2f}%")
        print(f"{archive}: {total_rows} lines, Failure Percentage: {failure_percentage:.2f}%")

    return top_archives, file_list, file_list_names

#This function get the longest interval of some directory
# Does not walks subfolders such as bbr and cubic
def get_dir_longest_interval(original_dir, saving_dir):
    for arquivo in os.listdir(original_dir):
        if arquivo.endswith('.csv'):
            print(f"Processando o arquivo: {arquivo}")
            caminho_arquivo = os.path.join(original_dir, arquivo)
            df = pd.read_csv(caminho_arquivo)

            longest_interval = []
            current_interval = []

            if 'Throughput' not in df.columns:
                print(f"A coluna 'Throughput' não foi encontrada no arquivo {arquivo}. Pulando este arquivo.")
                continue

            for index, row in df.iterrows():
                if pd.isna(row['Throughput']):  # Check if the value is NaN
                    # Update longest_interval if current_interval is greater
                    if len(current_interval) > len(longest_interval):
                        longest_interval = current_interval
                    current_interval = []
                else:
                    current_interval.append(row)  # Add the row to the current range

            # Check again at the end if the largest range ends at the end of the file
            if len(current_interval) > len(longest_interval):
                longest_interval = current_interval

            if longest_interval:
                print(f"Largest gap-free interval found in file {arquivo} with {len(longest_interval)} lines.")
                longest_df = pd.DataFrame(longest_interval)
                output_file = os.path.splitext(arquivo)[0] + '_longest_interval.csv'
                longest_df.to_csv(os.path.join(saving_dir, output_file), index=False)
            else:
                print(f"No continuous gapless intervals were found in the file {arquivo}.")

# This function checks the original dataset failure rate and applies in his respective longest interval
def apply_original_failure_rate_on_longest(original_dir, longest_interval_dir, saving_dir):
    for arquivo in os.listdir(original_dir):
        path = os.path.join(original_dir, arquivo)
        df = pd.read_csv(path)

        novo_nome = arquivo.replace('.csv', '_longest_interval.csv')
        path2 = os.path.join(longest_interval_dir, novo_nome)
        df2 = pd.read_csv(path2)

        # Calculate the percentage of missing data for the original dataset
        percentual_faltantes = df.isna().sum().sum() / (df.shape[0] * df.shape[1])

        # Calculate the number of values ​​to be made NaN in the 'Throughput' column
        total_celulas = df2.shape[0]  # Total rows in 'Throughput' column
        num_faltantes = int(total_celulas * percentual_faltantes)

        # Choose random indexes only on the 'Throughput' column
        nan_indices = np.random.choice(df2.index, num_faltantes, replace=False)
        df2.loc[nan_indices, 'Throughput'] = np.nan  

        path_salvar = os.path.join(saving_dir, novo_nome)
        df2.to_csv(path_salvar, index=False)
        print(f"{novo_nome} processed with NaNs applied in the column 'Throughput'")

In [ ]:
def plot_graph(df, link_name=""):
    plt.figure(figsize=(12, 6))
    plt.plot(df["Timestamp"], df["Throughput"], linestyle="-")
    plt.title(f"Network Performance {link_name}")
    plt.xlabel("Timestamp")
    plt.ylabel("Throughput (Mbps)")
    plt.tight_layout()
    plt.show()

def calculate_covariance_coef(df, link):
    df_cleaned = df.dropna(subset=['Throughput'])

    mean_throughput = df_cleaned['Throughput'].mean()
    std_dev_throughput = df_cleaned['Throughput'].std()

    cv_throughput = (std_dev_throughput / mean_throughput) * 100

    print(f"The Throughput Coefficient of Variation (CV) for {link} is: {cv_throughput:.2f}%")

def covariance_analysis(path):
    for file in os.listdir(path):
        file_path = os.path.join(path, file)
        df = pd.read_csv(file_path)
        plot_graph(df, file)
        calculate_covariance_coef(df, file)

Removing Unamed 0 column that was generated in some proccess

In [ ]:
diretorio = '../datasets/choosen-best-svd/'

# Iterar sobre todos os arquivos do diretório
for arquivo in os.listdir(diretorio):
    if arquivo.endswith('.csv'):
        caminho_arquivo = os.path.join(diretorio, arquivo)
        
        # Carregar o arquivo CSV, removendo a coluna "Unnamed: 0"
        df = pd.read_csv(caminho_arquivo)
        
        # Verificar se a coluna "Unnamed: 0" existe e removê-la
        if 'Unnamed: 0' in df.columns:
            df = df.drop(columns=['Unnamed: 0'])
            
            # Salvar o arquivo atualizado (substituir o original)
            df.to_csv(caminho_arquivo, index=False)
            print(f'Coluna "Unnamed: 0" removida do arquivo: {arquivo}')
        else:
            print(f'O arquivo "{arquivo}" não possui a coluna "Unnamed: 0".')
